In [1]:
"""
Web検索連動チャットボットを作成しよう

"""

# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver


# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    # ステートグラフの作成
    graph_builder = StateGraph(State)
    # 言語モデルの定義 
    llm = ChatOpenAI(model_name=model_name)
    # 検索ツールの定義 
    tool = TavilySearchResults(max_results=2) 
    tools = [tool]
    # ツールをLLMにバインド（ひもづけ）
    llm_with_tools = llm.bind_tools(tools)
    # チャットボットノードを定義 
    def chatbot(state: State): 
        return {"messages": [llm_with_tools.invoke(state["messages"])]}
    # グラフにチャットボットノードを追加
    graph_builder.add_node("chatbot", chatbot)
    # ツ－ルノードの作成
    tool_node = ToolNode(tools)
    # グラフにツールノードを追加
    graph_builder.add_node("tools", tool_node)
    # 条件付エッジの作成  
    graph_builder.add_conditional_edges(
        "chatbot",  #第一引数：遷移元（チャットボット　
        tools_condition, # ツール呼出と判断したらツールノードを呼ぶ
    )
    # ツールが呼び出されるたびに、チャットボットに戻って次のステップを決定
    # ツールからチャットボットへの戻りエッジを作成
    graph_builder.add_edge("tools", "chatbot")
    
    # 開始ノードの指定
    graph_builder.set_entry_point("chatbot")
    
    # 記憶を持つ実行可能なステートグラフの作成
    memory = MemorySaver() # チェックポインタ。記憶をもたせる。
    graph = graph_builder.compile(checkpointer=memory) 
    return graph
    #実行可能なグラフを返して、graphを作成


# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    # ソースコードを記述
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values")
    # 結果をストリーミングで得る
    for event in events:
        print(event["messages"][-1].content, flush=True)


# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
# ソースコードを記述
graph = build_graph(MODEL_NAME)

# メインループ
# ソースコードを記述
while True:
    user_input = input("質問:")
    if user_input.strip()=="":
        print("ありがとうございました!")
        break
    stream_graph_updates(graph, user_input)

こんばんは
こんばんは！今日はどのようなことをお手伝いできますか？
1たす2は？
1たす2は3です。何か他に計算や質問がありますか？
台湾観光についておしえて

[{"url": "https://tw-ryugaku.com/department_cat/leisure-tourism/", "content": "2000年に設立された台湾で最も新しい国立大学の一つです。自然に囲まれた広大で美しいキャンパスには、... ◇国立高雄科技大学高雄."}, {"url": "https://taiwan-talk.co.jp/taiwan-university/", "content": "歴史的背景  \n  台湾における科技大学の前身は、1950年代から1970年代にかけて設立された「工業専科學校）」です。当時、国家経済の発展と産業技術の向上を目的として、「臺灣工業専科學校（現在の国立台北科技大学）」、「高雄工専（現在の国立高雄科技大学）」などが設立されました。  \n  1990年代に入ると、産業の高度化と高等教育への需要の増加により、政府は職業技術教育の高度化政策を推進し、多くの専門学校が「技術学院」へと改組され、その後「科技大學」へと昇格しました。  \n  現在、台湾の科技大学は、工学、デザイン、経営、観光、ホスピタリティ、生物技術など多様な分野をカバーし、産業界で活躍できる実務力のある人材を育成しています。\n 意義  \n  科技大学の設立目的は、産業界が求める即戦力となる技術者や専門職人材の育成です。学生は在学中に実務を重視したカリキュラムに参加し、インターンシップ、プロジェクト型学習、企業との連携授業などを通じて、実際の職場で通用するスキルを身につけます。  \n  さらに、科技大学は企業との連携を強化し、「産学連携センター」「技術移転プラットフォーム」「イノベーション育成センター」などを設置しています。これにより、企業が抱える実務課題の解決に貢献し、学生もリアルなプロジェクトに参加する機会を得ています。  \n  このような教育体制は、いわゆる「学んだことが現場で通用しない」というギャップを大きく縮め、高等教育の新たなモデルとして注目されています。\n\n## 【2025年】台湾の文系大学ランキング\n\n3つ目は、文系大学ランキングです。 [.